# Main research notebook

Retained experiments and application prototype. Results in the paper have not been rerun. See docs/REPRODUCIBILITY.md before using Run All, especially the station/hour baseline and historical Granger-causality section.


In [ ]:
from pathlib import Path
import os

# Resolve the prepared repository from either its root or a notebook folder.
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data" / "processed" / "loader_f30avg.csv").is_file()
)
os.chdir(PROJECT_ROOT)
print("Working directory:", PROJECT_ROOT.name)


# Imports

In [ ]:
from capymoa.regressor import KNNRegressor, FIMTDD, AdaptiveRandomForestRegressor, SGDRegressor
from capymoa.stream import Schema, NumpyStream
from capymoa.instance import RegressionInstance
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from math import sqrt
from collections import defaultdict, deque
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
import capymoa

In [ ]:
capymoa.__version__

In this notebook I aim to go through all the naive approaches that were already done, and I'll build on them with some of my own ideas (+ look at a couple of applications).

In [ ]:
base_regressors_to_test = [KNNRegressor, FIMTDD, AdaptiveRandomForestRegressor]

# Naive Approaches

I tried the:
- Naive all per stop regressor
- Naive per stop regressor
- Naive regressor
- Naive chunked regressor

I wanted to try the timestamp naive chunked regressor but running time was too long.

## Naive All Per Stop Regressor

In [ ]:
class NaiveAllPerStopRegressor:
    def __init__(self, stop_ids, window_size=5,delay=6, base_regressor=FIMTDD):
        '''
        Inits regressor for naive approach

        Args:
            stop_ids (str[]): List of stop ids from file
            window_size (int): Number of lag features for model to predict on
        '''

        # Set window sizes and create
        self.window_size = window_size
        self.window = []
        # Create custom schema for dataset
        self.schema = Schema.from_custom(
            feature_names=[
                f"feature_{f}_t{-t}"
                for t in range(self.window_size)
                for f in range(len(stop_ids))
            ],
            dataset_name="OccupancyPredict",
            target_type="numeric"
        )

        self.delay = delay
        self.instance_buffer = {stop_id: deque(maxlen=self.delay) for stop_id in stop_ids}
            
        # Using FIMTDD as regressor
        self.regressors = {stop_id: base_regressor(self.schema) for stop_id in stop_ids}
        self.log = {stop_id:[] for stop_id in stop_ids}

    def update_window(self, value):
        """
        Updates regressor windows with an instance

        Args:
            value (int): value of instance
        """
        # Sliding window
        if len(self.window) == self.window_size:
            self.window.pop(0)
        self.window.append(value)

    def get_window(self):
        """
        Flatten the rows of data in window into a 1d array
        """
        return [item for sublist in self.window for item in sublist]

    def train(self, stop_id, target, features):
        """
        Trains regressor for stop_id on an instance

        Args:
            stop_id (str): String of stop_id data comes from
            target (float): Target value to predict
            features (array): Data to use to predict the value
        """

        # Update window
        self.update_window(features)

        # If there aren't enough instances to implement lag features can't train
        if len(self.window) < self.window_size:
            return
        
        # Build lag features and train regressor
        x = np.array(self.get_window())
        instance = RegressionInstance.from_array(self.schema, x, target)
        self.log[str(int(stop_id))].append({'x':x,'target':target})

        self.instance_buffer[str(int(stop_id))].append(instance)
        if len(self.instance_buffer[str(int(stop_id))]) == self.delay:
            self.regressors[str(int(stop_id))].train(self.instance_buffer[str(int(stop_id))][0])

    def predict(self, stop_id):
        """
        Predicts value

        Args:
            stop_id (str): String of stop_id to predict
        """

        # Can't predict without lag features
        if len(self.window) < self.window_size:
            return 0.0
        
        # Build lag features, predict
        x = np.array(self.get_window())
        instance = RegressionInstance.from_array(self.schema, x, 0.0) # Needs target
        return self.regressors[str(int(stop_id))].predict(instance)

In [ ]:
selected_model = NaiveAllPerStopRegressor

In [ ]:
# Load in csv as dataframe
df = pd.read_csv("data/processed/loader_f30avg.csv", index_col=0)

stop_ids = [
    "125960550",
    "230565994",
    "258781031",
    "43768720",
    "44072192",
    "44783654",
    "44783914",
    "44784438",
    "45833547",
    "47568123" ]
for base_regressor in base_regressors_to_test:
    print(base_regressor.__name__)
    # Init model
    model = selected_model(stop_ids, base_regressor=base_regressor)
    
    # Keeping track of errors
    total_abs_error = 0.0
    total_squared_error = 0.0
    count = 0
    
    # Window error tracking
    error_window = 100
    abs_error_buf = 0.0
    sqrd_error_buf = 0.0
    buf_count = 0
    mae_list = []
    rmse_list = []
    
    # Loop over values send to regressor as stream
    for _, row in tqdm(df.iterrows(), total=len(df)):
            
        y_value = row['f30avg']
    
        # Predict and get error
        pred = model.predict(row['stop'])
        error = pred - y_value
    
        # Update total errors
        total_abs_error += abs(error)
        total_squared_error += error ** 2
        count += 1
    
        # Update buf errors
        abs_error_buf += abs(error)
        sqrd_error_buf += error ** 2
        buf_count += 1
    
        # Train on instance
        model.train(row['stop'], y_value, row[stop_ids])
    
        # Add windowed error
        if buf_count == error_window:
            mae = abs_error_buf / error_window
            rmse = np.sqrt(sqrd_error_buf / error_window)
            mae_list.append(mae)
            rmse_list.append(rmse)
    
            # Reset
            abs_error_buf = 0.0
            sqrd_error_buf = 0.0
            buf_count = 0
    
    # Get final metrics
    mae = total_abs_error / count
    rmse = np.sqrt(total_squared_error / count)
    
    print(f"Final MAE: {mae:.3f}")
    print(f"Final RMSE: {rmse:.3f}")
    
    # Plot mae and rmse over windows
    plt.figure()
    plt.plot(mae_list, label="MAE")
    plt.plot(rmse_list, label="RMSE")
    plt.title("MAE and RMSE over time: "+selected_model.__name__)
    plt.legend()
    plt.grid(True)
    plt.ylim((0,100))
    plt.show()

In [ ]:
model.log['125960550'][0]

This has one model for every stop. Input is lag information about all stops from 5min data. Output is f30avg label.

## Naive Per Stop

In [ ]:
class NaivePerStopRegressor:
    def __init__(self, stop_ids, num_features=5, delay=6, base_regressor=FIMTDD):
        '''
        Inits regressor for naive approach

        Args:
            stop_ids (str[]): List of stop ids from file
            num_features (int): Number of lag features for model to predict on
        '''
        self.num_features = num_features
        # Set window sizes and create
        self.window_size = num_features
        self.windows = {stop_id: [] for stop_id in stop_ids}

        self.delay = delay
        self.instance_buffer = {stop_id: deque(maxlen=self.delay) for stop_id in stop_ids}
        
        # Create custom schema for dataset
        self.schema = Schema.from_custom(
            feature_names=[str(i) for i in range(num_features)],
            dataset_name="OccupancyPredict",
            target_type="numeric"
        )
        # Using FIMTDD as regressor
        self.regressors = {stop_id: base_regressor(self.schema) for stop_id in stop_ids}
        self.log = {stop_id: [] for stop_id in stop_ids}

    def update_window(self, stop_id, value):
        """
        Updates regressor winodws with an instance

        Args:
            stop_id (str): String of stop_id
            value (int): value of instance
        """
        window = self.windows[stop_id]
        # Sliding window
        if len(window) == self.window_size:
            window.pop(0)
        window.append(value)

    def train(self, stop_id, target, feature):
        """
        Trains regressor on an instance

        Args:
            stop_id (str): String of stop_id data comes from
            target (float): Target value to predict
        """
        # Update window
        self.update_window(stop_id, feature)

        window = self.windows[stop_id]
        # If there aren't enough instances to implement lag features can't train
        if len(window) < self.num_features:
            return
        
        # Build lag features and train regressor
        x = np.array(window)
        instance = RegressionInstance.from_array(self.schema, x, target)
        self.log[stop_id].append({'x':x,'target':target})

        self.instance_buffer[stop_id].append(instance)

        if len(self.instance_buffer[stop_id]) == self.delay:
            self.regressors[stop_id].train(self.instance_buffer[stop_id][0])

    def predict(self, stop_id):
        """
        Predicts value

        Args:
            stop_id (str): String of stop_id to predict from
        """
        window = self.windows[stop_id]
        # Can't predict without lag features
        if len(window) < self.num_features:
            return 0.0
        
        # Build lag features, predict
        x = np.array(window)
        instance = RegressionInstance.from_array(self.schema, x, 0.0) # Needs target
        return self.regressors[stop_id].predict(instance)

In [ ]:
selected_model = NaivePerStopRegressor

In [ ]:
# Load in csv as dataframe
df = pd.read_csv("data/processed/loader_f30avg_wide.csv", index_col=0)

# Get all stop ids
stop_ids = df.columns.to_list()

feature_ids = [
    "125960550",
    "230565994",
    "258781031",
    "43768720",
    "44072192",
    "44783654",
    "44783914",
    "44784438",
    "45833547",
    "47568123" ]

target_ids = [
    "125960550_f30avg",
    "230565994_f30avg",
    "258781031_f30avg",
    "43768720_f30avg",
    "44072192_f30avg",
    "44783654_f30avg",
    "44783914_f30avg",
    "44784438_f30avg",
    "45833547_f30avg",
    "47568123_f30avg"
    ]
for base_regressor in base_regressors_to_test:
    print(base_regressor.__name__)
    
    # Init model
    model = selected_model(target_ids,base_regressor=base_regressor)
    
    # Keeping track of errors
    total_abs_error = 0.0
    total_squared_error = 0.0
    count = 0
    
    # Window error tracking
    error_window = 100
    abs_error_buf = 0.0
    sqrd_error_buf = 0.0
    buf_count = 0
    mae_list = []
    rmse_list = []
    
    # Loop over values send to regressor as stream
    for _, row in tqdm(df.iterrows(), total=len(df)):
        for stop_id in target_ids:
    
            feature_value = row[stop_id.split("_")[0]]
    
            value = row[stop_id]
    
            # Predict and get error
            pred = model.predict(stop_id)
            error = pred - value
    
            # Update total errors
            total_abs_error += abs(error)
            total_squared_error += error ** 2
            count += 1
    
            # Update buf errors
            abs_error_buf += abs(error)
            sqrd_error_buf += error ** 2
            buf_count += 1
    
            # Train on instance
            model.train(stop_id, value, feature_value)
    
            # Add windowed error
            if buf_count == error_window:
                mae = abs_error_buf / error_window
                rmse = np.sqrt(sqrd_error_buf / error_window)
                mae_list.append(mae)
                rmse_list.append(rmse)
    
                # Reset
                abs_error_buf = 0.0
                sqrd_error_buf = 0.0
                buf_count = 0
    
    # Get final metrics
    mae = total_abs_error / count
    rmse = np.sqrt(total_squared_error / count)
    
    print(f"Final MAE: {mae:.3f}")
    print(f"Final RMSE: {rmse:.3f}")
    
    # Plot mae and rmse over windows
    plt.figure()
    plt.plot(mae_list, label="MAE")
    plt.plot(rmse_list, label="RMSE")
    plt.title("MAE and RMSE over time: "+selected_model.__name__)
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
model.log['125960550_f30avg'][0]

This just has the lags for a singular stop. Input is lag information from 5min stop data. Output is f30avg label.

## Naive Chunked

In [ ]:
class NaiveRegressorChunked:
    def __init__(self, stop_ids, num_features=5, base_regressor=base_regressor, delay=6):
        '''
        Inits regressor for naive approach

        Args:
            stop_ids (str[]): List of stop ids from file
            num_features (int): Number of lag features for model to predict on
        '''
        self.num_features = num_features
        # Set window sizes and create
        self.window_size = num_features
        self.windows = {stop_id: [] for stop_id in stop_ids}
        # Create custom schema for dataset
        self.schema = Schema.from_custom(
            feature_names=[str(i) for i in range(num_features)],
            dataset_name="OccupancyPredict",
            target_type="numeric"
        )
        self.delay = delay
        self.instance_buffer = {stop_id: deque(maxlen=delay) for stop_id in stop_ids}
        # Using FIMTDD as regressor
        self.regressor = base_regressor(self.schema)
        self.log = []

    def update_all_windows(self, row):
        """
        Updates all per-stop lag windows using current occupancy data

        Args:
            row (pd.Series): A row from dataframe of occupancies per stop
        """
        for stop_id in self.windows.keys():
            value = row[str(stop_id)]
            window = self.windows[stop_id]
            if len(window) == self.window_size:
                window.pop(0)
            window.append(value)

    def train(self, stop_id, target):
        """
        Trains regressor on an instance

        Args:
            stop_id (str): String of stop_id data comes from
            target (float): Target value to predict
        """
        window = self.windows[stop_id]
        # If there aren't enough instances to implement lag features can't train
        if len(window) < self.num_features:
            return
        
        # Build lag features and train regressor
        x = np.array(window)
        instance = RegressionInstance.from_array(self.schema, x, target)
        self.log.append({'x':x,'target':target})
        
        self.instance_buffer[stop_id].append(instance)
        
        if len(self.instance_buffer[stop_id]) == self.delay:
            self.regressor.train(self.instance_buffer[stop_id][0])                 

    def predict(self, stop_id):
        """
        Predicts value

        Args:
            stop_id (str): String of stop_id to predict from
        """
        window = self.windows[stop_id]
        # Can't predict without lag features
        if len(window) < self.num_features:
            return 0.0
        
        # Build lag features, predict
        x = np.array(window)
        instance = RegressionInstance.from_array(self.schema, x, 0.0) # Needs target
        return self.regressor.predict(instance)

This creates a shared autoregressive model. Takes five previous datapoints and predicts the next one. Does better than the others.

# New approaches

## Prequential evaluation

This is a prequential evaluation function that works with the 'naive chunked' data.

In [ ]:
def prequential_evaluation(selected_model, df, context_df=None, error_window=2400, plot=True, return_model = False, base_regressor=FIMTDD):
    """
    Custom prequential evaluation function with optional context features.

    Args:
        selected_model (class): The model class to evaluate.
        df (pd.DataFrame): Index must be timestamp. Must contain 'stop', 'f30avg', and stop_id columns.
        context_df (pd.DataFrame, optional): Must also be indexed by timestamp.
        error_window (int): Window size for rolling error tracking.
        plot (bool): Whether to plot MAE and RMSE over time.

    Returns:
        (float, float, list, list): Final MAE, final RMSE, MAE list, RMSE list
    """
    # Ensure datetime index
    df.index = pd.to_datetime(df.index)
    if context_df is not None:
        context_df.index = pd.to_datetime(context_df.index)

    # Get stop IDs
    stop_ids = [col for col in df.columns if col not in ["f30avg", "stop"]]

    # Init model
    model = selected_model(stop_ids, base_regressor=base_regressor)

    # Error tracking
    total_abs_error = 0.0
    total_squared_error = 0.0
    count = 0

    abs_error_buf = 0.0
    sqrd_error_buf = 0.0
    buf_count = 0
    mae_list = []
    rmse_list = []

    # Iterate over stream
    for timestamp, row in tqdm(df.iterrows(), total=len(df)):
        # Update lag windows
        model.update_all_windows(row)

        stop_id = str(int(row["stop"]))
        target = row["f30avg"]

        # Get context features for this timestamp
        context_features = context_df.loc[timestamp] if context_df is not None and timestamp in context_df.index else None

        # Predict
        if context_features is not None:
            pred = model.predict(stop_id, context_features)
        else:
            pred = model.predict(stop_id)

        error = pred - target

        # Update errors
        total_abs_error += abs(error)
        total_squared_error += error ** 2
        count += 1

        abs_error_buf += abs(error)
        sqrd_error_buf += error ** 2
        buf_count += 1

        # Train
        if context_features is not None:
            model.train(stop_id, target, context_features)
        else:
            model.train(stop_id, target)

        # Rolling window errors
        if buf_count == error_window:
            mae = abs_error_buf / error_window
            rmse = np.sqrt(sqrd_error_buf / error_window)
            mae_list.append(mae)
            rmse_list.append(rmse)
            abs_error_buf = 0.0
            sqrd_error_buf = 0.0
            buf_count = 0

    # Final metrics
    mae = total_abs_error / count
    rmse = np.sqrt(total_squared_error / count)

    print(f"Final MAE: {mae:.3f}")
    print(f"Final RMSE: {rmse:.3f}")

    if plot:
        plt.figure()
        plt.plot(mae_list, label="MAE")
        plt.plot(rmse_list, label="RMSE")
        plt.title("MAE and RMSE over time: " + selected_model.__name__)
        plt.legend()
        plt.grid(True)
        plt.show()

    if return_model:
        return mae, rmse, mae_list, rmse_list, model
    else:
        return mae, rmse, mae_list, rmse_list

In [ ]:
df = pd.read_csv("data/processed/loader_f30avg.csv", index_col=0)
for base_regressor in base_regressors_to_test:
    print(base_regressor.__name__)
    mae, rmse, mae_list, rmse_list = prequential_evaluation(NaiveRegressorChunked, df, base_regressor=base_regressor)

## Super Naive Model

In [ ]:
class SuperNaiveModel:
    def __init__(self, stop_ids, num_features=5, base_regressor=None):
        '''
        Trivial baseline regressor that predicts the mean of the most recent lag values.

        Args:
            stop_ids (str[]): List of stop IDs
            num_features (int): Number of lag values to average
        '''
        self.num_features = num_features
        self.window_size = num_features
        self.windows = {stop_id: [] for stop_id in stop_ids}
        self.log = []

    def update_all_windows(self, row):
        """
        Updates all per-stop lag windows with raw values.

        Args:
            row (pd.Series): A row from dataframe of occupancies per stop
        """
        for stop_id in self.windows:
            value = row[str(stop_id)]
            window = self.windows[stop_id]
            if len(window) == self.window_size:
                window.pop(0)
            window.append(value)

    def train(self, stop_id, target):
        """
        No training needed for this naive model, but log data for consistency/debugging.
        """
        window = self.windows[stop_id]
        self.log.append({'x': window.copy(), 'target': target})

    def predict(self, stop_id):
        """
        Predicts the average of the most recent lag values.

        Args:
            stop_id (str): Stop ID to predict for

        Returns:
            float: Mean of the lag values or 0.0 if not enough data
        """
        window = self.windows[stop_id]
        if len(window) < self.num_features:
            return 0.0
        return float(np.mean(window))

In [ ]:
df = pd.read_csv("data/processed/loader_f30avg.csv", index_col=0)
mae, rmse, mae_list, rmse_list = prequential_evaluation(SuperNaiveModel, df)

## Integrated Models

To make a truly integrated model:
- We want to predict the difference between the means and the next 30 minute average
- We predict this difference using the differences between the lags

In [ ]:
class NaiveRegressorIntegrated:
    def __init__(self, stop_ids, num_features=5, base_regressor=FIMTDD, delay=6, clip_factor=100):
        '''
        Differenced/integrated naive regressor.

        - Inputs: differences between consecutive values (length = num_features - 1)
        - Target: future value minus mean of recent lags
        - Prediction: predicted delta added to mean of recent lags

        Args:
            stop_ids (str[]): List of stop IDs
            num_features (int): Number of lag values to maintain
        '''
        self.num_features = num_features
        self.window_size = num_features
        self.windows = {stop_id: [] for stop_id in stop_ids}
        self.clip_factor = clip_factor
        self.delay = delay
        self.instance_buffer = {stop_id: deque(maxlen=delay) for stop_id in stop_ids}

        # Only use differences as features (length = num_features - 1)
        self.schema = Schema.from_custom(
            feature_names=[f"diff_{i}" for i in range(num_features - 1)],
            dataset_name="OccupancyPredictIntegrated",
            target_type="numeric"
        )

        self.regressor = base_regressor(self.schema)
        self.log = []

    def update_all_windows(self, row):
        """
        Updates per-stop lag windows with raw values (not differenced yet).

        Args:
            row (pd.Series): A row from the dataframe with all stop values
        """
        for stop_id in self.windows:
            value = row[str(stop_id)]
            window = self.windows[stop_id]
            if len(window) == self.window_size:
                window.pop(0)
            window.append(value)

    def train(self, stop_id, target):
        """
        Trains on the delta from the mean of the current lag window.

        Args:
            stop_id (str): Stop ID
            target (float): Actual observed target value
        """
        window = self.windows[stop_id]
        if len(window) < self.num_features:
            return

        # Compute inputs and target delta
        diff_lags = np.diff(window)  # shape: (num_features - 1,)
        mean_lag = np.mean(window)
        delta_target = target - mean_lag

        instance = RegressionInstance.from_array(self.schema, diff_lags, delta_target)
        self.log.append({'x': diff_lags.copy(), 'mean': mean_lag, 'target': target, 'delta': delta_target})
        
        self.instance_buffer[stop_id].append(instance)
        
        if len(self.instance_buffer[stop_id]) == self.delay:
            self.regressor.train(self.instance_buffer[stop_id][0])      

    def predict(self, stop_id):
        """
        Predicts the next value as: recent mean + predicted delta.

        Args:
            stop_id (str): Stop ID

        Returns:
            float: Predicted next value
        """
        window = self.windows[stop_id]
        if len(window) < self.num_features:
            return 0.0

        diff_lags = np.diff(window)
        mean_lag = np.mean(window)

        instance = RegressionInstance.from_array(self.schema, diff_lags, 0.0)
        predicted_delta = self.regressor.predict(instance)
        if predicted_delta > self.clip_factor or predicted_delta < -self.clip_factor:
            predicted_delta = 0
        return mean_lag + predicted_delta

In [ ]:
df = pd.read_csv("data/processed/loader_f30avg.csv", index_col=0)
for base_regressor in base_regressors_to_test:
    print(base_regressor.__name__)
    mae, rmse, mae_list, rmse_list = prequential_evaluation(NaiveRegressorIntegrated, df, base_regressor=base_regressor)

## Exogenous regressors

In [ ]:
df_contextual_data = pd.read_csv('data/processed/context_lagged.csv', index_col=0)

In [ ]:
class SeasonalRegressorIntegrated:
    def __init__(self, stop_ids, num_features=5, base_regressor=FIMTDD, delay=6, clip_factor=100):
        '''
        Seasonal integrated regressor using lag differences and cyclical time features.

        - Inputs: lag differences + time features (e.g., hour_sin, hour_cos)
        - Target: future value minus mean of recent lags
        - Prediction: predicted delta added to mean of recent lags
        '''
        self.num_features = num_features
        self.window_size = num_features
        self.windows = {stop_id: [] for stop_id in stop_ids}
        self.clip_factor = clip_factor
        self.delay = delay
        self.instance_buffer = {stop_id: deque(maxlen=delay) for stop_id in stop_ids}
        self.stop_ids = ['125960550','230565994','258781031','43768720','44072192','44783654','44783914','44784438','45833547','47568123']
        
        # Feature schema: lag diffs + 2 time features
        self.schema = Schema.from_custom(
            feature_names=[f"diff_{i}" for i in range(num_features - 1)] + ["hour_sin", "hour_cos"] + self.stop_ids,
            dataset_name="OccupancyPredictSeasonalIntegrated",
            target_type="numeric"
        )

        self.regressor = base_regressor(self.schema)
        self.log = []

    def update_all_windows(self, row):
        """Updates per-stop lag windows with raw values."""
        for stop_id in self.windows:
            value = row[str(stop_id)]
            window = self.windows[stop_id]
            if len(window) == self.window_size:
                window.pop(0)
            window.append(value)

    def train(self, stop_id, target, context_features):
        """
        Trains on the delta from the mean of the current lag window with time features.

        Args:
            stop_id (str): Stop ID
            target (float): Actual observed target value
            context_features (pd.Series): Must include hour_sin and hour_cos
        """
        window = self.windows[stop_id]
        if len(window) < self.num_features:
            return

        diff_lags = np.diff(window)  # shape: (num_features - 1,)
        mean_lag = np.mean(window)
        delta_target = target - mean_lag

        one_hot = np.zeros(len(self.stop_ids), dtype=int)
        index = self.stop_ids.index(stop_id)
        one_hot[index] = 1

        features = np.concatenate([diff_lags, context_features[["hour_sin", "hour_cos"]].values,one_hot])
        instance = RegressionInstance.from_array(self.schema, features, delta_target)

        self.log.append({'x': features.copy(), 'mean': mean_lag, 'target': target, 'delta': delta_target})
        self.instance_buffer[stop_id].append(instance)
        if len(self.instance_buffer[stop_id]) == self.delay:
            self.regressor.train(self.instance_buffer[stop_id][0])  

    def predict(self, stop_id, context_features):
        """
        Predicts the next value using lag window and time features.

        Args:
            stop_id (str): Stop ID
            context_features (pd.Series): Must include hour_sin and hour_cos

        Returns:
            float: Predicted next value
        """
        window = self.windows[stop_id]
        if len(window) < self.num_features:
            return 0.0

        diff_lags = np.diff(window)
        mean_lag = np.mean(window)

        one_hot = np.zeros(len(self.stop_ids), dtype=int)
        index = self.stop_ids.index(stop_id)
        one_hot[index] = 1
        

        features = np.concatenate([diff_lags, context_features[["hour_sin", "hour_cos"]].values,one_hot])
        instance = RegressionInstance.from_array(self.schema, features, 0.0)
        predicted_delta = self.regressor.predict(instance)
        if predicted_delta > self.clip_factor or predicted_delta < -self.clip_factor:
            predicted_delta = 0
        return mean_lag + predicted_delta

In [ ]:
df = pd.read_csv("data/processed/loader_f30avg.csv", index_col=0)

seasonal_integrated_models = {}

for base_regressor in base_regressors_to_test:
    print(base_regressor.__name__)
    mae, rmse, mae_list, rmse_list, model = prequential_evaluation(
        SeasonalRegressorIntegrated,
        df,
        context_df=df_contextual_data,
        return_model = True,
        base_regressor = base_regressor
    )
    seasonal_integrated_models[base_regressor.__name__] = {'model':model,'mae_list':mae_list, 'rmse_list':rmse_list}

In [ ]:
class TimeRegressorIntegrated:
    def __init__(self, stop_ids, num_features=5, base_regressor=FIMTDD, delay=6, clip_factor=100):
        '''
        Seasonal integrated regressor using lag differences and cyclical time features.

        - Inputs: lag differences + time features (e.g., hour_sin, hour_cos)
        - Target: future value minus mean of recent lags
        - Prediction: predicted delta added to mean of recent lags
        '''
        self.num_features = num_features
        self.window_size = num_features
        self.windows = {stop_id: [] for stop_id in stop_ids}
        self.clip_factor = clip_factor
        self.delay = delay
        self.instance_buffer = {stop_id: deque(maxlen=delay) for stop_id in stop_ids}
        self.stop_ids = ['125960550','230565994','258781031','43768720','44072192','44783654','44783914','44784438','45833547','47568123']

        # Feature schema: lag diffs + 2 time features
        self.schema = Schema.from_custom(
            feature_names=[f"diff_{i}" for i in range(num_features - 1)] + ["hour_sin", "hour_cos","Is_Weekend","Public_Holiday","Is_Event"] + self.stop_ids,
            dataset_name="OccupancyPredictSeasonalIntegrated",
            target_type="numeric"
        )

        self.regressor = base_regressor(self.schema)
        self.log = []

    def update_all_windows(self, row):
        """Updates per-stop lag windows with raw values."""
        for stop_id in self.windows:
            value = row[str(stop_id)]
            window = self.windows[stop_id]
            if len(window) == self.window_size:
                window.pop(0)
            window.append(value)

    def train(self, stop_id, target, context_features):
        """
        Trains on the delta from the mean of the current lag window with time features.

        Args:
            stop_id (str): Stop ID
            target (float): Actual observed target value
            context_features (pd.Series): Must include hour_sin and hour_cos
        """
        window = self.windows[stop_id]
        if len(window) < self.num_features:
            return

        diff_lags = np.diff(window)  # shape: (num_features - 1,)
        mean_lag = np.mean(window)
        delta_target = target - mean_lag

        one_hot = np.zeros(len(self.stop_ids), dtype=int)
        index = self.stop_ids.index(stop_id)
        one_hot[index] = 1

        features = np.concatenate([diff_lags, context_features[["hour_sin", "hour_cos","Is_Weekend","Public_Holiday","Is_Event"]].values, one_hot])
        instance = RegressionInstance.from_array(self.schema, features, delta_target)

        self.log.append({'x': features.copy(), 'mean': mean_lag, 'target': target, 'delta': delta_target})
        self.instance_buffer[stop_id].append(instance)
        if len(self.instance_buffer[stop_id]) == self.delay:
            self.regressor.train(self.instance_buffer[stop_id][0])  

    def predict(self, stop_id, context_features):
        """
        Predicts the next value using lag window and time features.

        Args:
            stop_id (str): Stop ID
            context_features (pd.Series): Must include hour_sin and hour_cos

        Returns:
            float: Predicted next value
        """
        window = self.windows[stop_id]
        if len(window) < self.num_features:
            return 0.0

        diff_lags = np.diff(window)
        mean_lag = np.mean(window)

        one_hot = np.zeros(len(self.stop_ids), dtype=int)
        index = self.stop_ids.index(stop_id)
        one_hot[index] = 1

        features = np.concatenate([diff_lags, context_features[["hour_sin", "hour_cos","Is_Weekend","Public_Holiday","Is_Event"]].values, one_hot])
        instance = RegressionInstance.from_array(self.schema, features, 0.0)
        predicted_delta = self.regressor.predict(instance)
        if predicted_delta > self.clip_factor or predicted_delta < -self.clip_factor:
            predicted_delta = 0
        return mean_lag + predicted_delta

In [ ]:
df = pd.read_csv("data/processed/loader_f30avg.csv", index_col=0)

for base_regressor in base_regressors_to_test:
    print(base_regressor.__name__)
    mae, rmse, mae_list, rmse_list = prequential_evaluation(
        TimeRegressorIntegrated,
        df,
        context_df=df_contextual_data,
        base_regressor=base_regressor
    )

In [ ]:
class ExogenousRegressorIntegrated:
    def __init__(self, stop_ids, num_features=5, base_regressor=FIMTDD, delay=6, clip_factor=100):
        '''
        Seasonal integrated regressor using lag differences and cyclical time features.

        - Inputs: lag differences + time features (e.g., hour_sin, hour_cos)
        - Target: future value minus mean of recent lags
        - Prediction: predicted delta added to mean of recent lags
        '''
        self.num_features = num_features
        self.window_size = num_features
        self.windows = {stop_id: [] for stop_id in stop_ids}
        self.clip_factor = clip_factor
        self.delay = delay
        self.instance_buffer = {stop_id: deque(maxlen=delay) for stop_id in stop_ids}
        self.stop_ids = ['125960550','230565994','258781031','43768720','44072192','44783654','44783914','44784438','45833547','47568123']

        # Feature schema: lag diffs + 2 time features
        self.schema = Schema.from_custom(
            feature_names=[f"diff_{i}" for i in range(num_features - 1)] + ['Day_Of_Week', 'Public_Holiday', 'Is_Event',
       'hour_sin', 'hour_cos', 'Is_Weekend', 'Flu_Like_Symptoms_Count',
       'Dengue_Count', 'ibov_open', 'temperature_2m', 'precipitation',
       'cloudcover', 'windspeed_10m', 'pm2_5', 'ozone',
       'us_aqi', 'onibus', 'metro', 'festival', 'trem', 'greve'] + self.stop_ids,
            dataset_name="OccupancyPredictSeasonalIntegrated",
            target_type="numeric"
        )

        self.regressor = base_regressor(self.schema)
        self.log = []

    def update_all_windows(self, row):
        """Updates per-stop lag windows with raw values."""
        for stop_id in self.windows:
            value = row[str(stop_id)]
            window = self.windows[stop_id]
            if len(window) == self.window_size:
                window.pop(0)
            window.append(value)

    def train(self, stop_id, target, context_features):
        """
        Trains on the delta from the mean of the current lag window with time features.

        Args:
            stop_id (str): Stop ID
            target (float): Actual observed target value
            context_features (pd.Series): Must include hour_sin and hour_cos
        """
        window = self.windows[stop_id]
        if len(window) < self.num_features:
            return

        diff_lags = np.diff(window)  # shape: (num_features - 1,)
        mean_lag = np.mean(window)
        delta_target = target - mean_lag

        one_hot = np.zeros(len(self.stop_ids), dtype=int)
        index = self.stop_ids.index(stop_id)
        one_hot[index] = 1

        features = np.concatenate([diff_lags, context_features[['Day_Of_Week', 'Public_Holiday', 'Is_Event',
       'hour_sin', 'hour_cos', 'Is_Weekend', 'Flu_Like_Symptoms_Count',
       'Dengue_Count', 'ibov_open', 'temperature_2m', 'precipitation',
       'cloudcover', 'windspeed_10m', 'pm2_5', 'ozone',
       'us_aqi', 'onibus', 'metro', 'festival', 'trem', 'greve']].values, one_hot])
        instance = RegressionInstance.from_array(self.schema, features, delta_target)

        self.log.append({'x': features.copy(), 'mean': mean_lag, 'target': target, 'delta': delta_target})
        self.instance_buffer[stop_id].append(instance)
        if len(self.instance_buffer[stop_id]) == self.delay:
            self.regressor.train(self.instance_buffer[stop_id][0])  

    def predict(self, stop_id, context_features):
        """
        Predicts the next value using lag window and time features.

        Args:
            stop_id (str): Stop ID
            context_features (pd.Series): Must include hour_sin and hour_cos

        Returns:
            float: Predicted next value
        """
        window = self.windows[stop_id]
        if len(window) < self.num_features:
            return 0.0

        diff_lags = np.diff(window)
        mean_lag = np.mean(window)

        one_hot = np.zeros(len(self.stop_ids), dtype=int)
        index = self.stop_ids.index(stop_id)
        one_hot[index] = 1

        features = np.concatenate([diff_lags, context_features[['Day_Of_Week', 'Public_Holiday', 'Is_Event',
       'hour_sin', 'hour_cos', 'Is_Weekend', 'Flu_Like_Symptoms_Count',
       'Dengue_Count', 'ibov_open', 'temperature_2m', 'precipitation',
       'cloudcover', 'windspeed_10m', 'pm2_5', 'ozone',
       'us_aqi', 'onibus', 'metro', 'festival', 'trem', 'greve']].values, one_hot])
        instance = RegressionInstance.from_array(self.schema, features, 0.0)
        predicted_delta = self.regressor.predict(instance)
        if predicted_delta > self.clip_factor or predicted_delta < -self.clip_factor:
                    predicted_delta = 0
        return mean_lag + predicted_delta

In [ ]:
df = pd.read_csv("data/processed/loader_f30avg.csv", index_col=0)

exogenous_models = {}

for base_regressor in base_regressors_to_test:
    print(base_regressor.__name__)
    mae, rmse, mae_list, rmse_list, model = prequential_evaluation(
        ExogenousRegressorIntegrated,
        df,
        context_df=df_contextual_data,
        return_model = True,
        base_regressor = base_regressor
    )
    exogenous_models[base_regressor.__name__] = {'model':model,'mae_list':mae_list, 'rmse_list':rmse_list}

# Pretty graphs

In [ ]:
import seaborn as sns

In [ ]:
mae_list = seasonal_integrated_models['FIMTDD']['mae_list']
rmse_list = seasonal_integrated_models['FIMTDD']['rmse_list']

# Downsample your datetime index to match the number of points
# Assuming 1 point per 100 datetimes in df.index
date_index = df.index[::100][:len(mae_list)]  # match the length of mae_list

# Create a DataFrame for plotting
plot_df = pd.DataFrame({
    'Date': date_index,
    'MAE': mae_list,
    'RMSE': rmse_list
})

# Melt to long-form for seaborn
plot_df_long = plot_df.melt(id_vars='Date', var_name='Metric', value_name='Error')

# Set Seaborn style
sns.set(style="whitegrid", palette="muted", font_scale=1.2)

# Plot
plt.figure(figsize=(12, 6))
sns.lineplot(data=plot_df_long, x='Date', y='Error', hue='Metric',)

# Titles and labels
plt.xlabel('Date')
plt.ylabel('Error')

# Tweak legend and layout
plt.legend(title='', loc='best')
plt.tight_layout()
plt.show()

## Station + Hour Time-Aware Baseline

In [ ]:
df = pd.read_csv("data/processed/loader_f30avg.csv")
df['datetime'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('datetime')
df['hour'] = df['datetime'].dt.hour
df['station'] = df['stop'].astype(str)

# Features y target
features = df[['hour', 'station']]
target = df['f30avg']

encoder = OneHotEncoder(sparse_output=False)
X_encoded = encoder.fit_transform(features)
X_array = X_encoded.astype(np.float32)
y_array = target.to_numpy().astype(np.float32)

def evaluate_model(model_class, name, X_base, y_base):
    stream = NumpyStream(np.copy(X_base), np.copy(y_base), dataset_name=name)
    model = model_class(stream.get_schema())

    true_vals, preds = [], []
    buf_X, buf_y = [], []

    while stream.has_more_instances():
        inst = stream.next_instance()
        y_i = inst.y_value

        if len(buf_X) >= 6:
            inst_pred = buf_X.pop(0)
            pred = model.predict(inst_pred)
            true_vals.append(buf_y.pop(0))
            preds.append(pred)

        model.train(inst)
        buf_X.append(inst)
        buf_y.append(y_i)

    mae = mean_absolute_error(true_vals, preds)
    rmse = sqrt(mean_squared_error(true_vals, preds))
    print(f"{name} — MAE: {mae:.2f}, RMSE: {rmse:.2f}")

evaluate_model(FIMTDD, "FIMTDD", X_array, y_array)
evaluate_model(KNNRegressor, "KNN", X_array, y_array)
evaluate_model(AdaptiveRandomForestRegressor, "ARF", X_array, y_array)


# Application 1: Feature Importances

In [ ]:
stop_ids = ['125960550','230565994','258781031','43768720','44072192','44783654','44783914','44784438','45833547','47568123']

ex_schema = Schema.from_custom(
            feature_names=[f"diff_{i}" for i in range(5 - 1)] + ['Day_Of_Week', 'Public_Holiday', 'Is_Event',
       'hour_sin', 'hour_cos', 'Is_Weekend', 'Flu_Like_Symptoms_Count',
       'Dengue_Count', 'ibov_open', 'temperature_2m', 'precipitation',
       'cloudcover', 'windspeed_10m', 'pm2_5', 'ozone',
       'us_aqi', 'onibus', 'metro', 'festival', 'trem', 'greve'] + stop_ids,
            dataset_name="OccupancyPredictSeasonalIntegrated",
            target_type="numeric"
        )

def predict_with_model(model,x):
    instance = RegressionInstance.from_array(ex_schema, x, 0.0)
    return model.regressor.predict(instance)

In [ ]:
class StreamingFeatureAttributionArray:
    def __init__(self, model, predict_fn, feature_names, baseline_strategy="zero", perturb_fn=None):
        """
        Args:
            model: Trained CapyMOA model (not used directly, passed to predict_fn)
            predict_fn: Function (model, x_array) -> prediction
            feature_names: List of feature names in the same order as array input
            baseline_strategy: "zero" or "mean"
            perturb_fn: Optional function to perturb values (value, feature_name) -> new_value
        """
        self.model = model
        self.predict_fn = predict_fn
        self.feature_names = feature_names
        self.baseline_strategy = baseline_strategy
        self.perturb_fn = perturb_fn

        self.n_features = len(feature_names)
        if baseline_strategy == "mean":
            self.feature_counts = np.zeros(self.n_features)
            self.feature_sums = np.zeros(self.n_features)
        else:
            self.feature_counts = None
            self.feature_sums = None

    def _get_feature_baseline(self, i):
        """
        Return the current baseline value for feature i
        """
        if self.baseline_strategy == "zero":
            return 0.0
        elif self.baseline_strategy == "mean":
            if self.feature_counts[i] == 0:
                return 0.0  # fallback for first time
            return self.feature_sums[i] / self.feature_counts[i]
        else:
            raise NotImplementedError(f"Unknown baseline strategy: {self.baseline_strategy}")

    def explain(self, x):
        """
        Compute per-feature attribution by perturbing each feature.

        Args:
            x (np.ndarray): 1D input array with shape (n_features,)

        Returns:
            dict: feature_name -> importance score
        """
        x = np.asarray(x, dtype=np.float64)

        # Update running means before perturbation (important!)
        if self.baseline_strategy == "mean":
            self.feature_counts += 1
            self.feature_sums += x

        base_pred = self.predict_fn(self.model, x)

        attributions = {}
        for i, name in enumerate(self.feature_names):
            x_perturbed = x.copy()

            if self.perturb_fn:
                x_perturbed[i] = self.perturb_fn(x[i], name)
            else:
                x_perturbed[i] = self._get_feature_baseline(i)

            perturbed_pred = self.predict_fn(self.model, x_perturbed)
            attributions[name] = base_pred - perturbed_pred

        return attributions

In [ ]:
model = exogenous_models['AdaptiveRandomForestRegressor']['model']

feature_names = [f"diff_{i}" for i in range(5 - 1)] + ['Day_Of_Week', 'Public_Holiday', 'Is_Event',
       'hour_sin', 'hour_cos', 'Is_Weekend', 'Flu_Like_Symptoms_Count',
       'Dengue_Count', 'ibov_open', 'temperature_2m', 'precipitation',
       'cloudcover', 'windspeed_10m', 'pm2_5', 'ozone',
       'us_aqi', 'onibus', 'metro', 'festival', 'trem', 'greve'] + stop_ids

explainer = StreamingFeatureAttributionArray(
    model= model,
    predict_fn=predict_with_model,
    feature_names=feature_names,
    baseline_strategy='mean'
)
all_explanations = []

# Iterate over each row in the dataset
for idx, dict_ in tqdm(enumerate(model.log), total=len(model.log)):
    x = dict_['x']
    explanation = explainer.explain(x)
    explanation["__row__"] = idx  # tag row index for reference
    all_explanations.append(explanation)

# Convert to DataFrame: rows = instances, columns = feature importances
explanations_df = pd.DataFrame(all_explanations).set_index("__row__")

In [ ]:
explanations_df.to_parquet('explanations.parquet')

In [ ]:
mean_importance = explanations_df.mean().abs().sort_values(ascending=False)

print("Mean absolute feature importances:")
print(mean_importance)

In [ ]:
weather_features = ['temperature_2m', 'windspeed_10m', 'precipitation', 'cloudcover']

# Melt the DataFrame into long format
plot_df = explanations_df[weather_features].reset_index().melt(
    id_vars="__row__", var_name="Feature", value_name="Importance"
)

# Use absolute value before smoothing
plot_df['AbsImportance'] = plot_df['Importance'].abs()

# Optional smoothing on the absolute values
plot_df['SmoothedAbs'] = plot_df.groupby('Feature')['AbsImportance'].transform(
    lambda x: x.rolling(window=2400, min_periods=1).mean()
)

# Plot smoothed absolute importance
plt.figure(figsize=(12, 6))
sns.lineplot(data=plot_df, x="__row__", y="SmoothedAbs", hue="Feature")
plt.title("Smoothed Absolute Weather Feature Importances Over Time")
plt.xlabel("Timestep (row index)")
plt.ylabel("Smoothed |Attribution|")
plt.legend(title="Weather Feature", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Application 2: Granger Causality

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests

In [ ]:
df_contextual_data_not_lagged = pd.read_csv('data/processed/context_not_lagged.csv')[240:].reset_index(drop=True)

In [ ]:
df = pd.read_csv("data/processed/loader_f30avg.csv", index_col=0)

mae, rmse, mae_list, rmse_list_seasonal_integrated, seasonal_integrated_fimtdd = prequential_evaluation(
    SeasonalRegressorIntegrated,
    df,
    context_df=df_contextual_data,
    return_model = True,
    error_window=2400,
    base_regressor=FIMTDD
)

In [ ]:
df = df_contextual_data_not_lagged.copy()
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df.set_index('Timestamp', inplace=True)
columns = ['Flu_Like_Symptoms_Count','Dengue_Count','onibus','metro','festival','trem','greve']
daily_df = df[columns].resample('1D').mean()
daily_df = daily_df.reset_index()
daily_df = daily_df[:-1]

In [ ]:
daily_df['Model_RMSE'] = rmse_list_seasonal_integrated

In [ ]:
maxlag = 14

In [ ]:
daily_df

In [ ]:
def grangers_causation_matrix(data, variables, test='ssr_chi2test', verbose=False):    
    """Check Granger Causality of all possible combinations of the Time series.
    The rows are the response variable, columns are predictors. The values in the table 
    are the P-Values. P-Values lesser than the significance level (0.05), implies 
    the Null Hypothesis that the coefficients of the corresponding past values is 
    zero, that is, the X does not cause Y can be rejected.

    data      : pandas dataframe containing the time series variables
    variables : list containing names of the time series variables.

    Citation: https://www.machinelearningplus.com/time-series/granger-causality-test-in-python/
    """
    df = pd.DataFrame(np.zeros((len(variables), len(variables))), columns=variables, index=variables)
    for c in df.columns:
        for r in df.index:
            test_result = grangercausalitytests(data[[r, c]], maxlag=maxlag, verbose=False)
            p_values = [round(test_result[i+1][0][test][1],4) for i in range(maxlag)]
            if verbose: print(f'Y = {r}, X = {c}, P Values = {p_values}')
            min_p_value = np.min(p_values)
            df.loc[r, c] = min_p_value
    df.columns = [var + '_x' for var in variables]
    df.index = [var + '_y' for var in variables]
    return df

In [ ]:
granger_matrix = grangers_causation_matrix(daily_df,['Flu_Like_Symptoms_Count', 'Dengue_Count', 'onibus',
       'metro', 'festival', 'trem', 'greve', 'Model_RMSE'])

In [ ]:
granger_matrix < 0.05/(64) #Bonferroni correction

The error of the model granger-causes flu-symptoms and dengue (and trem) -- suggesting potential causation between public transport volumes and these diseases.

Searches for 'onibus', 'metro' and 'greve' granger-cause the error of the model.

# Application 3: Busy station detection

Busy station detection

In [ ]:
base_regressor=FIMTDD
selected_model=SeasonalRegressorIntegrated
stop_ids = ['125960550','230565994','258781031','43768720','44072192','44783654','44783914','44784438','45833547','47568123']
station_data = pd.read_csv("data/reference/bus_stops.csv")
station_data.index = [str(int(i)) for i in station_data['stop_id']]
model = selected_model(stop_ids, base_regressor=FIMTDD)

In [ ]:
def get_station_coordinates(stop_id):
    """
    Get geographical coordinates of a station
    """
    if stop_id in station_data.index:
        station_info = station_data.loc[stop_id]
        return {
            'lat': station_info['stop_lat'],
            'lon': station_info['stop_lon'],
            'name': station_info['stop_name']
        }
    else:
        return None

In [ ]:
def get_all_station_coordinates():
    """
    Get geographical coordinates of all stations
    """
    coordinates = {}
    for station_id in stop_ids:
        if str(int(station_id)) in station_data.index:
            station_info = station_data.loc[str(int(station_id))]
            coordinates[str(int(station_id))] = {
                    'lat': station_info['stop_lat'],
                    'lon': station_info['stop_lon'],
                    'name': station_info['stop_name']
                }
    return coordinates

In [ ]:
station_locations = get_all_station_coordinates()

In [ ]:
def calculate_distance(station1, station2):
    """
    Calculate distance between two stations (simplified Euclidean distance)
    """
    station1_location = station_locations[station1]
    station2_location = station_locations[station2]
        
    # Calculate distance
    distance = np.sqrt((station1_location["lat"]-station2_location["lat"])**2 + (station1_location["lon"]-station2_location["lon"])**2) * 111000  # Convert to meters
    return distance

In [ ]:
recommendation_radius = 10000

In [ ]:
def get_nearby_stations(target_station):
    """
    Get nearby stations within specified radius
    """
    nearby_stations = []
        
    for station in stop_ids:
        if station != target_station:
            distance = calculate_distance(target_station, station)
            if distance <= recommendation_radius:
                nearby_stations.append((station, distance))
                    
    # Sort by distance
    nearby_stations.sort(key=lambda x: x[1])
    return [station for station, _ in nearby_stations]

In [ ]:
def suitability_score(busyness_orig, busyness_alt, distance_m, alpha=10000):
    """Higher is better. Negative if alternative is worse (busier)."""
    busyness_diff = busyness_orig - busyness_alt  # Positive if alternative is less busy
    return busyness_diff - (distance_m / alpha)

In [ ]:
def recommend_alternative_stations(busy_station, busyness_indices):
    """
    Recommend alternative stations for busy stations
    """
    busyness_index_of_station = busyness_indices[busy_station]
    # Get nearby stations
    nearby_stations = get_nearby_stations(busy_station)

    if not nearby_stations:
        return []
            
    # Predict future passenger flow for nearby stations
    recommendations = []
    for station in nearby_stations[:5]:  # Consider 5 closest stations
        # Predict future flow 
        busyness_index_of_alternative = busyness_indices[station]
        distance = calculate_distance(busy_station, station)
        score = suitability_score(busyness_index_of_station,busyness_index_of_alternative,distance)

        if score > 0:
            recommendations.append({
                'station': station,
                'busyness_index': busyness_index_of_alternative,
                'distance': distance,
                'score': score  # Better score for closer distance and lower flow
            })
                
            # Sort by score (higher score is better)
            recommendations.sort(key=lambda x: -x['score'])

    if len(recommendations) > 0:
        return recommendations[0]  # Return best recommendations
    else:
        return {}

In [ ]:
def busyness_index(z, load, z_thresh=1.5, load_thresh=100, z_slope=3, load_slope=0.05):
    z_component = 1 / (1 + np.exp(-z_slope * (z - z_thresh)))
    load_component = 1 / (1 + np.exp(-load_slope * (load - load_thresh)))
    return z_component * load_component

In [ ]:
example_time_range = df.index.unique()[9960:10021] # get output for this time range

In [ ]:
buffer = {stop_id: deque(maxlen=6) for stop_id in stop_ids} # buffer for delayed labelling
statistics = {stop_id: {'sum':0,'squared_sum':0,'count':0} for stop_id in stop_ids}

prev_timestamp = None
busy_stations = []
busyness_indices = {stop_id: None for stop_id in stop_ids}

simulation_results = {}
# Index of this is timestamp
# At each timestamp, can access busynesss_index and busy_stations
# Busy_stations is a dict of busy stations with dicts

for timestamp, row in tqdm(df.iterrows(), total=len(df)):

    # if at the end of timestamp, we can do the logic for the station recommendation system
    if timestamp != prev_timestamp:
        if len(busy_stations) > 0:
            busy_station_names = [station_locations[str(station)]['name'] for station in busy_stations[:3]]

            if prev_timestamp in example_time_range:
                print(f"Detected {len(busy_stations)} busy stations at {prev_timestamp}: {busy_station_names}")
            
            recommendations = {}

            for busy_station in busy_stations:
                rec = recommend_alternative_stations(busy_station, busyness_indices)
                if rec:
                    recommendations[busy_station] = rec
                    if prev_timestamp in example_time_range:
                        print(f"For station {station_locations[busy_station]['name']} - {busy_station} \nRecommended alternative station: {station_locations[rec['station']]['name']} - {rec['station']} \nDistance: {rec['distance']:.0f}m")
            
            simulation_results[timestamp] = {'busyness':busyness_indices.copy(),'busy_stations':recommendations}
        else:
            simulation_results[timestamp] = {'busyness':busyness_indices.copy(),'busy_stations':{}}
        # reset
        busy_stations = []
            
    
    
    # logic for test then train
    model.update_all_windows(row)
    stop_id = str(int(row['stop']))
    target = row["f30avg"]
    buffer[stop_id].append(row[stop_id])
    context_features = df_contextual_data.loc[timestamp]

    pred = model.predict(stop_id, context_features)
    model.train(stop_id, target, context_features)

    
    # only attempt to detect a busy station if buffer is full
    if len(buffer[stop_id]) == 6:
        # calculate rolling summary statistics
        new_data_point = buffer[stop_id][0]
        statistics[stop_id]['sum'] += new_data_point
        statistics[stop_id]['squared_sum'] += new_data_point**2
        statistics[stop_id]['count'] += 1
        
        n = statistics[stop_id]['count']
        sum_x = statistics[stop_id]['sum']
        sum_x2 = statistics[stop_id]['squared_sum']

        mean = sum_x / n
        variance = (sum_x2 / n) - (mean ** 2)
        sd = variance ** 0.5

        # detect busy station if z_score > 1.5 and pred > 100
        z_score = (pred - mean) / (sd + 1e-6)
        is_busy = z_score > 1.5 and pred > 100

        busyness_indices[stop_id] = busyness_index(z_score,pred)
        if is_busy:
            busy_stations.append(stop_id)            
        
    prev_timestamp = timestamp   
    

In [ ]:
station_coordinates = get_all_station_coordinates()

In [ ]:
def plot_station_busyness(timestamp, station_coordinates, simulation_results):
    busyness = simulation_results[timestamp]['busyness']
    
    lats = []
    lons = []
    colors = []
    
    for stop_id in busyness:
        if stop_id in station_coordinates:
            lats.append(station_coordinates[stop_id]['lat'])
            lons.append(station_coordinates[stop_id]['lon'])
            colors.append(busyness[stop_id])  # Color by busyness index (0 to 1)

    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(lons, lats, c=colors, cmap='viridis', s=80, edgecolor='k')
    plt.colorbar(scatter, label='Busyness Index')
    plt.title(f'Station Busyness - {timestamp}')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.grid(True)
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import numpy as np

def save_busyness_animation_gif(
    station_coordinates, simulation_results, timestamps, filename="outputs/busyness.gif"
):
    fig, ax = plt.subplots(figsize=(10, 8))

    # Calculate axis limits from all station coordinates once
    all_lats = [station_coordinates[s]['lat'] for s in station_coordinates]
    all_lons = [station_coordinates[s]['lon'] for s in station_coordinates]

    lat_margin = 0.1 * (max(all_lats) - min(all_lats))
    lon_margin = 0.1 * (max(all_lons) - min(all_lons))

    ax.set_xlim(min(all_lons) - lon_margin, max(all_lons) + lon_margin)
    ax.set_ylim(min(all_lats) - lat_margin, max(all_lats) + lat_margin)

    first_ts = timestamps[0]
    first_busyness = simulation_results[first_ts]['busyness']
    first_busy_stations = simulation_results[first_ts].get('busy_stations', {})

    lats, lons, colors = [], [], []
    busy_lats, busy_lons = [], []

    for stop_id in first_busyness:
        if stop_id in station_coordinates:
            lat = station_coordinates[stop_id]['lat']
            lon = station_coordinates[stop_id]['lon']
            lats.append(lat)
            lons.append(lon)
            colors.append(first_busyness[stop_id])
            if stop_id in first_busy_stations:
                busy_lats.append(lat)
                busy_lons.append(lon)

    if not lats:
        lats, lons, colors = [0], [0], [0]
    if not busy_lats:
        busy_lats, busy_lons = [0], [0]

    # Base scatter plot for all stations (colored by busyness)
    scatter = ax.scatter(
        lons, lats, c=colors, cmap='viridis', vmin=0, vmax=1,
        s=80, edgecolor='k', linewidth=0.5
    )

    # Red outline for busy stations
    busy_scatter = ax.scatter(
        busy_lons, busy_lats, facecolors='none', edgecolors='red',
        s=120, linewidth=2, label='Busy station'
    )

    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.grid(True)

    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Busyness Index')
    ax.legend(loc='upper right')

    # We'll keep references to arrows so we can remove them on update
    arrows = []

    def update(frame):
        nonlocal arrows

        # Clear old arrows
        for arrow in arrows:
            arrow.remove()
        arrows = []

        timestamp = timestamps[frame]
        busyness = simulation_results[timestamp]['busyness']
        busy_stations = simulation_results[timestamp].get('busy_stations', {})

        lats, lons, colors = [], [], []
        busy_lats, busy_lons = [], []

        for stop_id in busyness:
            if stop_id in station_coordinates:
                lat = station_coordinates[stop_id]['lat']
                lon = station_coordinates[stop_id]['lon']
                lats.append(lat)
                lons.append(lon)
                colors.append(busyness[stop_id])
                if stop_id in busy_stations:
                    busy_lats.append(lat)
                    busy_lons.append(lon)

        if lats:
            scatter.set_offsets(np.column_stack((lons, lats)))
            scatter.set_array(np.array(colors))
        if busy_lats:
            busy_scatter.set_offsets(np.column_stack((busy_lons, busy_lats)))
        else:
            busy_scatter.set_offsets(np.empty((0, 2)))

        # Draw arrows from busy station to alternative station
        for stop_id, info in busy_stations.items():
            if 'station' in info:
                alt_id = info['station']
                if stop_id in station_coordinates and alt_id in station_coordinates:
                    start_lat = station_coordinates[stop_id]['lat']
                    start_lon = station_coordinates[stop_id]['lon']
                    end_lat = station_coordinates[alt_id]['lat']
                    end_lon = station_coordinates[alt_id]['lon']

                    arrow = ax.annotate(
                        '',
                        xy=(end_lon, end_lat), xycoords='data',
                        xytext=(start_lon, start_lat), textcoords='data',
                        arrowprops=dict(arrowstyle='->', color='blue', lw=1.5)
                    )
                    arrows.append(arrow)

        ax.set_title(f'Station Recommendation System – {timestamp}')
        return scatter, busy_scatter, *arrows

    anim = FuncAnimation(
        fig, update, frames=len(timestamps), interval=1000, repeat=False
    )

    writer = PillowWriter(fps=1)
    anim.save(filename, writer=writer)
    print(f"Saved animation to: {filename}")


In [ ]:
save_busyness_animation_gif(station_coordinates, simulation_results, example_time_range)